# Análisis de Reseñas de `www.fanatical.com`

**Proyecto 6 – NLP | Master en Data Science & IA**

Fanatical es una plataforma de distribución digital de videojuegos, socia oficial de más de 1.200 publishers. Ofrece bundles, keys de Steam y descuentos en juegos PC.

Objetivos del análisis:
1. Sentimiento global de Fanatical vs competencia
2. Topics principales de las reseñas
3. Sentimiento por topic
4. Comparativa con el sector
5. Conclusiones y áreas de mejora

## 1) Librerías

In [ ]:
# pip install pandas scikit-learn matplotlib seaborn wordcloud transformers torch

import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from transformers import pipeline
from wordcloud import WordCloud

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print('Librerías cargadas ✓')

## 2) Configuración

In [ ]:
TARGET    = 'www.fanatical.com'
CAT       = 'Events & Entertainment'
N_TOPICS  = 6
TOP_WORDS = 12
DATA_PATH = '../CSV/trustpilot-reviews-123k.csv'
SENTIMENT_MODEL = 'distilbert-base-uncased-finetuned-sst-2-english'

# Paleta de colores centralizada
C_TARGET  = '#FF6B35'
C_POS     = '#4CAF50'
C_NEG     = '#F44336'
C_SECTOR  = '#90CAF9'

# Etiquetas de topics (asignadas tras inspeccionar palabras clave del NMF)
TOPIC_LABELS = {
    0: 'Mystery Bundles',
    1: 'Problemas de Pago / Antifraude',
    2: 'Experiencia de Compra Positiva',
    3: 'Atención al Cliente / Incidencias',
    4: 'Fidelización / Usuarios Recurrentes',
    5: 'Precios y Pagos',
}

# Stopwords adicionales específicas del dominio (complementan las de sklearn)
DOMAIN_STOPS = [
    'fanatical', 'game', 'games', 'bought', 'buy', 'purchase', 'purchased',
    'site', 'website', 'store', 'shop', 'got', 'good', 'great', 'use',
    'time', 'just', 'like', 'really', 'also', 've', 'didn', 'don', 'll',
]

print(f'Empresa : {TARGET}')
print(f'Sector  : {CAT}')

## 3) Carga y limpieza de datos

> **Nota:** El dataset está balanceado artificialmente — todas las empresas tienen exactamente 20 reseñas por cada estrella (media = 3.0★). El análisis NLP del texto refleja mejor la realidad.

In [ ]:
def load_sector(path: str, category: str, min_reviews: int = 100) -> pd.DataFrame:
    """Carga el CSV, filtra empresas con exactamente `min_reviews` reseñas
    y devuelve solo las del sector indicado."""
    df = pd.read_csv(path)
    counts = df.groupby('company').size()
    eligible = counts[counts == min_reviews].index
    return df[df['company'].isin(eligible) & (df['category'] == category)].copy()


def clean_text(text: str) -> str:
    """Normaliza una reseña para NLP: elimina ruido, pasa a minúsculas."""
    if not isinstance(text, str):
        return ''
    text = re.sub(r'https?://\S+|www\.\S+', '', text)        # elimina URLs
    text = re.sub(r'[@#]\w+', '', text)                       # elimina menciones y hashtags
    text = text.encode('ascii', errors='ignore').decode()      # elimina emojis y caracteres no-ASCII
    text = re.sub(r'[^\w\s\'\-\.,!?]', ' ', text)            # elimina caracteres especiales, conservando puntuación básica
    return re.sub(r'\s+', ' ', text).strip().lower()          # colapsa espacios y pasa a minúsculas


df_sector = load_sector(DATA_PATH, CAT)
df_sector['review_clean'] = df_sector['review'].map(clean_text)

# Concatenamos título + reseña: el título suele condensar la opinión
# principal del usuario y enriquece el vocabulario para el TF-IDF
df_sector['text_full'] = df_sector['title'].map(clean_text) + ' ' + df_sector['review_clean']

# Descartamos reseñas que quedaron vacías o casi vacías tras la limpieza
df_sector = df_sector[df_sector['review_clean'].str.len() > 10].reset_index(drop=True)

print(f'Reseñas en el sector : {len(df_sector):,}')
print(f'Empresas             : {df_sector["company"].nunique()}')
print(f'Reseñas de Fanatical : {(df_sector["company"] == TARGET).sum()}')

## 4) Exploración inicial

In [ ]:
df_fan = df_sector[df_sector['company'] == TARGET].copy()

print('=== Distribución de estrellas – Fanatical ===')
print(df_fan['stars'].value_counts().sort_index())

review_len = df_fan['review'].str.len()
print(f'\nLongitud reseñas: media={review_len.mean():.0f} | '
      f'min={review_len.min()} | max={review_len.max()} caracteres')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

review_len.clip(upper=review_len.quantile(0.98)).hist(
    bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set(title='Longitud de reseñas (caracteres)', xlabel='Caracteres')

df_fan['stars'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='coral', edgecolor='white', rot=0)
axes[1].set(title='Distribución de estrellas – Fanatical',
            xlabel='Estrellas', ylabel='Nº reseñas')

plt.tight_layout()
plt.show()

In [ ]:
for stars, label in [(5, 'positivas'), (1, 'negativas')]:
    print(f'=== Muestra de reseñas {label} ({stars}★) ===')
    for _, r in df_fan[df_fan['stars'] == stars].head(3).iterrows():
        print(f'  [{r["stars"]}★] {r["title"]}')
        print(f'  {str(r["review"])[:220]}\n')

## 5) Análisis de Sentimiento

**DistilBERT** fine-tuned en SST-2 clasifica cada reseña como `POSITIVE` / `NEGATIVE`.
Se trunca a 512 tokens (límite del modelo). La primera ejecución descarga ~250 MB del modelo.

In [ ]:
%%time
print('Cargando modelo de sentimiento...')

# DistilBERT es una versión comprimida de BERT (40% menos parámetros, 97% del rendimiento).
# Fine-tuned en SST-2 (Stanford Sentiment Treebank): dataset de críticas de cine en inglés.
# Devuelve POSITIVE / NEGATIVE con una puntuación de confianza entre 0 y 1.
sentiment_pipe = pipeline(
    'sentiment-analysis',
    model=SENTIMENT_MODEL,
    truncation=True,
    max_length=512,   # límite de tokens del modelo; las reseñas más largas se truncan
)

print('Ejecutando inferencia...')

# batch_size=32: procesa 32 reseñas a la vez para aprovechar la paralelización de la GPU/CPU
results = sentiment_pipe(
    df_sector['review_clean'].tolist(),
    batch_size=32,
    truncation=True,
    max_length=512,
)

df_sector['sentiment']   = [r['label'] for r in results]
df_sector['is_positive'] = df_sector['sentiment'] == 'POSITIVE'  # columna booleana para operar fácilmente

# Sincronizamos df_fan con las nuevas columnas calculadas sobre df_sector
df_fan = df_sector[df_sector['company'] == TARGET].copy()

print(f'\nSentimiento calculado para {len(df_sector):,} reseñas ✓')
print(df_sector['sentiment'].value_counts())

In [ ]:
def sentiment_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula % positivo por empresa, ordenado de menor a mayor (para el barh)."""
    return (
        df.groupby('company')['is_positive']
        .agg(total='count', positivas='sum')
        .assign(pct_pos=lambda x: x['positivas'] / x['total'] * 100)
        .sort_values('pct_pos', ascending=True)
        .reset_index()
    )


sent_sector = sentiment_summary(df_sector)

# .item() extrae el escalar Python del Series de un solo elemento (más seguro que .values[0])
fan_pct     = sent_sector.loc[sent_sector['company'] == TARGET, 'pct_pos'].item()
sector_mean = sent_sector['pct_pos'].mean()

# tolist().index() da la posición 0-indexed; sumamos 1 para el ranking legible
fan_rank = sent_sector['company'].tolist().index(TARGET) + 1
n_comp   = len(sent_sector)

pos = df_fan['is_positive'].sum()
neg = (~df_fan['is_positive']).sum()

print(f'Fanatical   : {fan_pct:.1f}% positivo  ({pos} pos / {neg} neg)')
print(f'Media sector: {sector_mean:.1f}%')
print(f'Ranking     : #{fan_rank} de {n_comp}')

In [ ]:
def bar_color(companies, target, c_target, c_default):
    return [c_target if c == target else c_default for c in companies]


fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pastel Fanatical
_, _, autotexts = axes[0].pie(
    [pos, neg],
    labels=['Positivo', 'Negativo'],
    autopct='%1.1f%%',
    colors=[C_POS, C_NEG],
    startangle=90,
    textprops={'fontsize': 13},
    pctdistance=0.75,
)
for at in autotexts:
    at.set(fontsize=13, fontweight='bold')
axes[0].set_title('Sentimiento Global\nFanatical.com', fontsize=14, fontweight='bold')
axes[0].text(0, -1.3, f'Ranking: #{fan_rank} de {n_comp} en el sector',
             ha='center', fontsize=10, color='gray')

# Barras por empresa
axes[1].barh(
    sent_sector['company'],
    sent_sector['pct_pos'],
    color=bar_color(sent_sector['company'], TARGET, C_TARGET, C_SECTOR),
    edgecolor='white',
)
axes[1].axvline(sector_mean, color='navy', linestyle='--', alpha=0.6,
                label=f'Media sector ({sector_mean:.0f}%)')
axes[1].set(xlabel='% Reseñas positivas',
            title=f'Sentimiento por empresa\n{CAT}')
axes[1].tick_params(axis='y', labelsize=7)
axes[1].legend(fontsize=9)
for lbl in axes[1].get_yticklabels():
    if TARGET in lbl.get_text():
        lbl.set(color=C_TARGET, fontweight='bold')

plt.tight_layout()
plt.show()

## 6) Modelado de Topics (NMF)

**TF-IDF + NMF** sobre las reseñas de Fanatical para extraer los 6 temas principales.
El mismo vectorizador se aplica luego al sector completo para hacer la comparativa.

| Topic | Etiqueta | Palabras clave |
|---|---|---|
| 0 | Mystery Bundles | mystery, bundle, winter, aaa, titles, misleading |
| 1 | Problemas de Pago / Antifraude | manual, authorization, flagged, frustrating, payment |
| 2 | Experiencia de Compra Positiva | easy, keys, worked, great deals, immediately |
| 3 | Atención al Cliente / Incidencias | refund, issue, support, key, steam, email |
| 4 | Fidelización / Usuarios Recurrentes | bundles, years, multiple, overall, purchases |
| 5 | Precios y Pagos | prices, fraud, card, bank, price, buying |

In [ ]:
# ── TF-IDF ───────────────────────────────────────────────────────────────────
# TF-IDF (Term Frequency - Inverse Document Frequency) convierte cada reseña
# en un vector numérico donde el peso de cada palabra refleja su relevancia:
# alta si aparece mucho en esa reseña pero poco en el resto del corpus.
#
# Parámetros clave:
#   max_features=3000  → vocabulario limitado a las 3000 palabras más relevantes
#   ngram_range=(1,2)  → incluye unigramas ("key") y bigramas ("steam key")
#                        para capturar expresiones compuestas
#   min_df=2           → ignora palabras que aparecen en menos de 2 reseñas (ruido)
#   max_df=0.85        → ignora palabras que aparecen en más del 85% de reseñas (genéricas)
tfidf = TfidfVectorizer(
    max_features=3000,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.85,
)

# Entrenamos el vectorizador SOLO sobre Fanatical para que los topics
# reflejen el vocabulario propio de esta empresa, no el del sector completo.
# Luego aplicaremos el mismo vectorizador al sector para comparar.
X_fan = tfidf.fit_transform(df_fan['text_full'])
vocab = np.array(tfidf.get_feature_names_out())

# Eliminamos stopwords de dominio del vocabulario resultante.
# No se pueden pasar directamente a TfidfVectorizer porque son términos
# compuestos o que sklearn no elimina con su lista estándar.
domain_mask = ~np.isin(vocab, DOMAIN_STOPS)
vocab_f     = vocab[domain_mask]

# ── NMF ──────────────────────────────────────────────────────────────────────
# NMF (Non-negative Matrix Factorization) factoriza la matriz TF-IDF en:
#   W (doc × topic): qué tanto pertenece cada reseña a cada topic
#   H (topic × term): qué palabras definen cada topic
#
# Ventaja sobre LDA: más rápido y produce topics más interpretables
# en datasets pequeños como este (100 reseñas).
nmf   = NMF(n_components=N_TOPICS, random_state=42, max_iter=300)
W_fan = nmf.fit_transform(X_fan[:, domain_mask])

print('=== Palabras clave por topic ===\n')
for i, row in enumerate(nmf.components_):
    # argsort()[::-1] ordena de mayor a menor peso; tomamos las TOP_WORDS primeras
    top_words = ', '.join(vocab_f[row.argsort()[::-1][:TOP_WORDS]])
    print(f'Topic {i} [{TOPIC_LABELS[i]}]:\n  {top_words}\n')

In [ ]:
def assign_topics(df: pd.DataFrame, tfidf, nmf, domain_mask, labels: dict) -> pd.DataFrame:
    """Aplica el vectorizador y el modelo NMF ya entrenados a un DataFrame
    y asigna a cada reseña el topic con mayor peso (argmax de la fila en W)."""
    # transform() (no fit_transform) usa el vocabulario aprendido en Fanatical
    W = nmf.transform(tfidf.transform(df['text_full'])[:, domain_mask])
    return df.assign(
        topic_id=W.argmax(axis=1),                              # índice numérico del topic dominante
        topic_label=pd.Series(W.argmax(axis=1)).map(labels).values,  # nombre legible del topic
    )


# Asignamos topics a Fanatical y al sector completo con el mismo modelo
# para que los topics sean comparables entre empresas
df_fan    = assign_topics(df_fan,    tfidf, nmf, domain_mask, TOPIC_LABELS)
df_sector = assign_topics(df_sector, tfidf, nmf, domain_mask, TOPIC_LABELS)

print('Distribución de topics en Fanatical:')
print(df_fan['topic_label'].value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, (ax, row) in enumerate(zip(axes.flat, nmf.components_)):
    # nmf.components_[i] contiene el peso de cada término en el topic i.
    # Usamos esos pesos directamente como frecuencias: a mayor peso, mayor tamaño en la nube.
    freqs = dict(zip(vocab_f, row))
    wc = WordCloud(
        width=400, height=250,
        background_color='white',
        colormap='Blues',
        max_words=40,
    ).generate_from_frequencies(freqs)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(TOPIC_LABELS[i], fontsize=11, fontweight='bold')

plt.suptitle('Word Clouds por Topic – Fanatical.com', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7) Sentimiento por Topic

In [ ]:
def topic_sentiment(df: pd.DataFrame) -> pd.DataFrame:
    """% positivo por topic, ordenado de mayor a menor."""
    return (
        df.groupby('topic_label')['is_positive']
        .agg(total='count', positivas='sum')
        .assign(pct_pos=lambda x: x['positivas'] / x['total'] * 100)
        .sort_values('pct_pos', ascending=False)
        .reset_index()
    )


sent_topic = topic_sentiment(df_fan)

print('=== Sentimiento por Topic – Fanatical ===')
for _, r in sent_topic.iterrows():
    icon = '✅' if r['pct_pos'] >= 50 else '❌'
    print(f'  {icon} {r["topic_label"]}: {r["pct_pos"]:.0f}%  ({r["total"]} reseñas)')

In [ ]:
plot_df = sent_topic.sort_values('pct_pos')   # ascendente para barh

fig, ax = plt.subplots(figsize=(10, 5))
colors = [C_POS if p >= 50 else C_NEG for p in plot_df['pct_pos']]
bars   = ax.barh(plot_df['topic_label'], plot_df['pct_pos'], color=colors, edgecolor='white')

ax.axvline(50, color='black', linestyle='--', alpha=0.4)
for bar, val in zip(bars, plot_df['pct_pos']):
    ax.text(val + 1, bar.get_y() + bar.get_height() / 2,
            f'{val:.0f}%', va='center', fontsize=11, fontweight='bold')

ax.set(xlabel='% Reseñas positivas', xlim=(0, 115),
       title='Sentimiento por Topic – Fanatical.com')
ax.legend(handles=[
    mpatches.Patch(color=C_POS, label='Satisfacción ≥ 50%'),
    mpatches.Patch(color=C_NEG, label='Satisfacción < 50%'),
], fontsize=9)
plt.tight_layout()
plt.show()

## 8) Comparativa con la Competencia

In [ ]:
# Calculamos el % positivo por topic para Fanatical y para el sector completo
# usando la misma función → resultado comparable en la misma escala
fan_by_topic    = topic_sentiment(df_fan).set_index('topic_label')['pct_pos']
sector_by_topic = topic_sentiment(df_sector).set_index('topic_label')['pct_pos']

comp_df = (
    pd.concat([fan_by_topic.rename('Fanatical'),
               sector_by_topic.rename('Sector (media)')], axis=1)
    # La diferencia positiva indica que Fanatical supera la media del sector en ese topic
    .assign(diferencia=lambda x: x['Fanatical'] - x['Sector (media)'])
    .sort_values('diferencia', ascending=False)   # primero las fortalezas
    .reset_index()
    .rename(columns={'topic_label': 'topic'})
)

print('=== Fanatical vs Sector ===')
for _, r in comp_df.iterrows():
    estado = 'MEJOR ✅' if r['diferencia'] >= 0 else 'PEOR  ❌'
    print(f'  [{estado}] {r["topic"]}')
    print(f'             Fanatical {r["Fanatical"]:.0f}% | '
          f'Sector {r["Sector (media)"]:.0f}% | Dif: {r["diferencia"]:+.1f} pp')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Barras agrupadas
x, width = np.arange(len(comp_df)), 0.35
axes[0].bar(x - width/2, comp_df['Fanatical'],      width, label='Fanatical', color=C_TARGET, edgecolor='white')
axes[0].bar(x + width/2, comp_df['Sector (media)'], width, label='Sector',    color=C_SECTOR, edgecolor='white')
axes[0].axhline(50, color='gray', linestyle='--', alpha=0.6)
axes[0].set_xticks(x)
axes[0].set_xticklabels(comp_df['topic'], rotation=18, ha='right', fontsize=8)
axes[0].set(ylabel='% Reseñas positivas', ylim=(0, 110),
            title='% Positivo por Topic\nFanatical vs Sector')
axes[0].legend()

# Diferencial
colors_diff = [C_POS if d >= 0 else C_NEG for d in comp_df['diferencia']]
axes[1].barh(comp_df['topic'], comp_df['diferencia'], color=colors_diff, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=1)
for i, val in enumerate(comp_df['diferencia']):
    axes[1].text(
        val + (1.5 if val >= 0 else -1.5), i,
        f'{val:+.1f} pp', va='center',
        ha='left' if val >= 0 else 'right',
        fontsize=9, fontweight='bold',
    )
axes[1].set(xlabel='Diferencia pp (Fanatical – Sector)',
            title='¿En qué supera Fanatical al sector?\n(verde = mejor que competencia)')

plt.tight_layout()
plt.show()

## 9) Heatmap: Sentimiento por Empresa y Topic

In [ ]:
# Construimos una tabla pivote: filas = empresas, columnas = topics, valores = % positivo
pivot = (
    df_sector.groupby(['company', 'topic_label'])['is_positive']
    .mean()
    .unstack()          # convierte el índice 'topic_label' en columnas
    .fillna(0)          # si una empresa no tiene reseñas en un topic, ponemos 0
                        # (evita NaN que rompería fmt='.0f' del heatmap)
    .mul(100)           # de proporción (0-1) a porcentaje (0-100)
)

# Ordenamos las empresas de mejor a peor sentimiento global
pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    pivot, annot=True, fmt='.0f', cmap='RdYlGn',
    vmin=0, vmax=100, linewidths=0.5,
    annot_kws={'size': 7}, ax=ax,
)
ax.set(title=f'% Reseñas Positivas por Empresa y Topic – {CAT}',
       xlabel='Topic', ylabel='')
ax.tick_params(axis='y', labelsize=7)
ax.tick_params(axis='x', rotation=20, labelsize=8)

# Resaltamos la fila de Fanatical para localizarla rápidamente
fan_row = list(pivot.index).index(TARGET)
ax.get_yticklabels()[fan_row].set(color=C_TARGET, fontweight='bold')

plt.tight_layout()
plt.show()

## 10) Ejemplos de reseñas negativas por topic

Para entender concretamente qué falla en cada área.

In [ ]:
# Mostramos las reseñas negativas ordenando por los topics con peor sentimiento primero
# → útil para identificar rápidamente las quejas más críticas
topics_ordered = sent_topic.sort_values('pct_pos')['topic_label'].tolist()

for topic in topics_ordered:
    neg = df_fan[(df_fan['topic_label'] == topic) & (~df_fan['is_positive'])]
    if neg.empty:
        continue
    pct = sent_topic.loc[sent_topic['topic_label'] == topic, 'pct_pos'].item()
    print(f'--- {topic}  ({len(neg)} negativas | {pct:.0f}% positivo) ---')
    for _, r in neg.head(2).iterrows():
        print(f'  [{r["stars"]}★] {str(r["review"])[:260]}')
    print()

## 11) Conclusiones y Áreas de Mejora

In [ ]:
mejoras  = comp_df[comp_df['diferencia'] < 0].sort_values('diferencia')
fortalezas = comp_df[comp_df['diferencia'] >= 0]

print('=' * 65)
print('  CONCLUSIONES – ANÁLISIS DE RESEÑAS FANATICAL.COM')
print('=' * 65)
print(f"""
1. SENTIMIENTO GLOBAL
   Fanatical   : {fan_pct:.0f}% positivo
   Media sector: {sector_mean:.0f}%
   Ranking     : #{fan_rank} de {n_comp} empresas en {CAT}
   → Ligeramente por debajo del sector, impulsado por problemas
     sistémicos concretos (antifraude y mystery bundles).
""")

print('2. FORTALEZAS (Fanatical supera al sector)')
for _, r in fortalezas.iterrows():
    print(f'   ✅ {r["topic"]}: {r["diferencia"]:+.1f} pp')

print('\n3. ÁREAS DE MEJORA (Fanatical por debajo del sector)')
for _, r in mejoras.iterrows():
    print(f'   ❌ {r["topic"]}: {r["diferencia"]:+.1f} pp')

print("""
4. RECOMENDACIONES
   • Antifraude (-21 pp): Revisar el sistema de autorización manual.
     Bloquea compras legítimas de usuarios recurrentes. Es el topic
     con 0% positivo, el peor del análisis.
   • Mystery Bundles (-12 pp): Garantizar una calidad mínima y
     mejorar la transparencia del contenido antes de la compra.
   • Precios (-10 pp): Comunicar mejor el valor real vs Steam.
   • Soporte (-8 pp): Agilizar resolución de incidencias con keys.

5. NOTA METODOLÓGICA
   El dataset está balanceado artificialmente (20 reseñas por
   estrella por empresa). El análisis NLP del texto es más
   representativo de la realidad que la distribución de estrellas.
""")
print('=' * 65)